# Financial News Sentiment Prediction using Deep Learning & BERT

This notebook builds a finance tweet sentiment classifier with baseline RNN models and an optional BERT-based model. It includes data loading from Hugging Face, preprocessing, training, evaluation, and inference.

## 1. Install and import libraries

In [ ]:
# !pip install -q numpy pandas scikit-learn matplotlib seaborn streamlit torch torchvision transformers datasets sentencepiece jupyter tqdm

In [ ]:
import os
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

## 2. Load dataset from Hugging Face

In [ ]:
dataset = load_dataset('zeroshot/twitter-financial-news-sentiment')
dataset

## 3. Inspect and prepare the data

In [ ]:
def dataset_to_df(split):
    df = pd.DataFrame(dataset[split])
    df = df.rename(columns={'text': 'tweet', 'label': 'label'})
    return df[['tweet', 'label']]

train_df = dataset_to_df('train')
validation_df = dataset_to_df('validation')
train_df.head()

In [ ]:
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
train_df['sentiment'] = train_df['label'].map(label_map)
validation_df['sentiment'] = validation_df['label'].map(label_map)
train_df['sentiment'].value_counts(), validation_df['sentiment'].value_counts()

## 4. Preprocessing and tokenization

In [ ]:
import re

def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^A-Za-z0-9$%&()*+,\-/.:;?@#\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

train_df['clean_text'] = train_df['tweet'].apply(clean_tweet)
validation_df['clean_text'] = validation_df['tweet'].apply(clean_tweet)
train_df.head()

In [ ]:
from collections import Counter

def build_vocab(texts, min_freq=2, max_size=15000):
    tokens = [token for text in texts for token in text.split()]
    freq = Counter(tokens)
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for idx, (token, count) in enumerate(freq.most_common(max_size), start=2):
        if count < min_freq:
            break
        vocab[token] = idx
    return vocab

vocab = build_vocab(train_df['clean_text'].tolist(), min_freq=2, max_size=15000)
len(vocab)

In [ ]:
def encode_text(text, vocab, max_len=50):
    tokens = text.split()
    encoded = [vocab.get(token, vocab['<UNK>']) for token in tokens][:max_len]
    padding = [vocab['<PAD>']] * (max_len - len(encoded))
    return encoded + padding

max_len = 50
train_df['input_ids'] = train_df['clean_text'].apply(lambda x: encode_text(x, vocab, max_len))
validation_df['input_ids'] = validation_df['clean_text'].apply(lambda x: encode_text(x, vocab, max_len))
train_df[['clean_text', 'input_ids']].head()

## 5. Dataset and DataLoader

In [ ]:
class FinancialTweetsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

batch_size = 64
train_dataset = FinancialTweetsDataset(train_df['input_ids'].tolist(), train_df['label'].tolist())
valid_dataset = FinancialTweetsDataset(validation_df['input_ids'].tolist(), validation_df['label'].tolist())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

## 6. Baseline model definitions

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, model_type='rnn', dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if model_type == 'rnn':
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        elif model_type == 'lstm':
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        elif model_type == 'gru':
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        else:
            raise ValueError('Unsupported model_type')
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.rnn(embedded)
        if isinstance(hidden, tuple):
            hidden = hidden[0]
        hidden = hidden[-1]  # last layer
        dropped = self.dropout(hidden)
        return self.fc(dropped)

## 7. Training and evaluation helpers

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    for texts, labels in loader:
        texts = texts.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * texts.size(0)
        all_preds.extend(predictions.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(loader.dataset), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro')

def eval_model(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in loader:
            texts = texts.to(device)
            labels = labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item() * texts.size(0)
            all_preds.extend(predictions.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(loader.dataset), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro'), all_labels, all_preds

## 8. Train baseline models

In [ ]:
def run_baseline(model_type, epochs=10):
    model = SentimentRNN(len(vocab), 128, 128, 3, model_type=model_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    history = []
    best_val_f1 = 0
    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, _, _ = eval_model(model, valid_loader, criterion)
        history.append((train_loss, train_acc, train_f1, val_loss, val_acc, val_f1))
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f'best_{model_type}.pt')
        print(f'Epoch {epoch+1}/{epochs} | Train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f} | Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}')
    return model, history

rnn_model, rnn_history = run_baseline('rnn', epochs=5)
lstm_model, lstm_history = run_baseline('lstm', epochs=5)
gru_model, gru_history = run_baseline('gru', epochs=5)

## 9. Baseline evaluation

In [ ]:
def report_results(model, loader, criterion, model_name):
    val_loss, val_acc, val_f1, labels, preds = eval_model(model, loader, criterion)
    print(f'{model_name} | Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}')
    print(classification_report(labels, preds, target_names=['Bearish', 'Bullish', 'Neutral']))
    sns.heatmap(confusion_matrix(labels, preds), annot=True, fmt='d', cmap='Blues', xticklabels=['Bearish', 'Bullish', 'Neutral'], yticklabels=['Bearish', 'Bullish', 'Neutral'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(model_name + ' Confusion Matrix')
    plt.show()

criterion = nn.CrossEntropyLoss()
report_results(rnn_model, valid_loader, criterion, 'Simple RNN')
report_results(lstm_model, valid_loader, criterion, 'LSTM')
report_results(gru_model, valid_loader, criterion, 'GRU')

## 10. Optional: BERT fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

bert_model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

def tokenize_batch(batch):
    return tokenizer(batch['tweet'], truncation=True, padding='max_length', max_length=64)

bert_dataset = load_dataset('zeroshot/twitter-financial-news-sentiment')
bert_dataset = bert_dataset.map(tokenize_batch, batched=True)
bert_dataset = bert_dataset.remove_columns(['text'])
bert_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=3)
training_args = TrainingArguments(
    output_dir='bert_output',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='bert_logs',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro'),
    }

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_dataset['train'],
    eval_dataset=bert_dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## 11. Inference helper

In [ ]:
def predict_text_rnn(model, text, vocab, max_len=50):
    model.eval()
    tokens = clean_tweet(text).split()
    input_ids = encode_text(' '.join(tokens), vocab, max_len)
    with torch.no_grad():
        outputs = model(torch.tensor([input_ids], dtype=torch.long).to(device))
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
        label = np.argmax(probs)
    return label_map[label], {label_map[i]: float(probs[i]) for i in range(len(probs))}

def predict_text_bert(text):
    encoded = tokenizer(text, truncation=True, padding='max_length', max_length=64, return_tensors='pt')
    encoded = {k: v.to(device) for k, v in encoded.items()}
    bert_model.to(device)
    bert_model.eval()
    with torch.no_grad():
        outputs = bert_model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        label = np.argmax(probs)
    return label_map[label], {label_map[i]: float(probs[i]) for i in range(len(probs))}

## 12. Save model artifacts

In [ ]:
torch.save({'model_state_dict': lstm_model.state_dict(), 'vocab': vocab}, 'best_lstm_baseline.pt')
print('Saved baseline model and vocab.')
import json
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f)
